In [2]:
import json

In [4]:
with open("../../data/retacred/test.json") as file:
  data = json.load(file)

In [7]:
# get the sequence lengths
seq_lens = []
for d in data:
  seq_lens.append((d['docid'], len(d['token'])))
  
seq_lens = sorted(list(set(seq_lens)))
print(len(seq_lens))
print(seq_lens[0])

5371
('00c7c29f9710890abe2f185bf0427589', 9)


In [9]:
num_docs = len(seq_lens)

bottom_perc = 0.15
top_perc = 0.85

bottom_ids = []
mid_ids = []
top_ids = []
# get the IDs
for i, (did, dtok) in enumerate(seq_lens):
    if i / num_docs <= bottom_perc:
        bottom_ids.append(did)
    elif i / num_docs >= top_perc:
        top_ids.append(did)
    else:
        mid_ids.append(did)

print(len(bottom_ids)/ num_docs)
print(len(mid_ids)/ num_docs)
print(len(top_ids)/ num_docs)


0.15006516477378515
0.7000558555203873
0.1498789797058276


In [13]:
bottom_test = []
mid_test = []
top_test = []

for d in data:
  if d['docid'] in bottom_ids:
      bottom_test.append(d)
  elif d['docid'] in mid_ids:
      mid_test.append(d)
  elif d['docid'] in top_ids:
      top_test.append(d)
  else:
      print("error")
      
print(len(bottom_test))
print(len(mid_test))
print(len(top_test))

1761
9773
1884


In [11]:
# write new json files for top, mid, and bottom test
with open("../../data/retacred/test_bottom.json", "w") as file:
  json.dump(bottom_test, file)
with open("../../data/retacred/test_mid.json", "w") as file:
  json.dump(mid_test, file)
with open("../../data/retacred/test_top.json", "w") as file:
  json.dump(top_test, file)

In [14]:
# CD into the parent folder for convinience

%cd ../../src_ra_cgcn
%pwd

/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn


'/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn'

In [15]:
# imports

import random
import argparse

from tqdm import tqdm
import torch

from data.loader import DataLoader, DataLoaderPredict
from model.trainer import GCNTrainer
from utils import torch_utils, scorer, constant, helper
from utils.vocab import Vocab


In [16]:
def load_model(model_dir, model="best_model.pt", seed=1234, cuda=None):
    # set cuda and cuda seed
    if cuda is None:
        cuda = torch.cuda.is_available()
        torch.cuda.manual_seed(seed)

    # set the seeds
    torch.manual_seed(seed)
    random.seed(seed)

    # load opt
    model_file = f"{model_dir}/{model}"
    print(f"Loading model from {model_file}")
    opt = torch_utils.load_config(model_file)
    trainer = GCNTrainer(opt)
    trainer.load(model_file)

    # load vocab
    vocab_file = f"{model_dir}/vocab.pkl"
    vocab = Vocab(vocab_file, load=True)
    assert opt['vocab_size'] == vocab.size, "Vocab size must match that in the saved model."

    return trainer, vocab, opt

In [17]:
def evaluate_model(trainer, vocab, data_dir="../data/retacred", opt=None, scorer=None, test_file="test.json"):
    assert opt is not None, "opt must not be None."
    assert scorer is not None, "scorer must not be None."

    # load data
    data_file = f"{data_dir}/{test_file}"
    print(f"Loading data from {data_file} with batch size {opt['batch_size']}...")
    batch = DataLoader(data_file, opt['batch_size'], opt, vocab, evaluation=True)

    label2id = constant.LABEL_TO_ID
    id2label = dict([(v,k) for k,v in label2id.items()])

    predictions = []
    all_probs = []
    batch_iter = tqdm(batch)
    for i, b in enumerate(batch_iter):
        preds, probs, _ = trainer.predict(b)
        predictions += preds
        all_probs += probs

    predictions = [id2label[p] for p in predictions]
    _, details = scorer.score(batch.gold(), predictions, verbose=False)

    # print("Detailed evaluation:")
    # print("\n".join([f"{k}: {v}" for k, v in details.items()]))
    print("Evaluation ended.")

In [18]:
trainer_agcn, vocab_agcn, opt_agcn = load_model(model_dir="saved_models/400/", model="best_model.pt")
trainer_gcn, vocab_gcn, opt_gcn = load_model(model_dir="saved_models/500/", model="best_model.pt")


Loading model from saved_models/400//best_model.pt
[ Fail: model loading failed. ]


/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn/utils/torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental

UnboundLocalError: local variable 'dump' referenced before assignment

In [ ]:
# eval for attention GCN (bottom)
evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file="test_bottom.json")

In [ ]:
# eval for attention GCN (mid)
evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file="test_mid.json")

In [ ]:
# eval for attention GCN (top)
evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file="test_top.json")

In [ ]:
# eval for GCN (bottom)
evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file="test_bottom.json")

In [ ]:
# eval for GCN (mid)
evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file="test_mid.json")

In [ ]:
# eval for GCN (top)
evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file="test_top.json")